In [1]:
import ee, folium, geemap, os

from xml.dom import minidom
import numpy as np
from sklearn.preprocessing import StandardScaler

#my helper here
# parse_kml_coordinates already imported above

# The main idea of this notebook is extract the embeddings
# more info about embeddings here:
# https://developers.google.com/machine-learning/crash-course/embeddings/
# https://developers.google.com/machine-learning/crash-course/embeddings/embedding-space
# something defenitivily worthy to study:
# https://deepmind.google.com/science/weatherlab

# 1. Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='eastern-thinker-471320-h4')
print("Earth Engine initialized.")


ModuleNotFoundError: No module named 'ee'

In [ ]:
# 2. Parse KML polygon for AOI
def parse_kml_coordinates(kml_path):
    """Parse KML polygon into list of [lon, lat] lists."""
    doc = minidom.parse(kml_path)
    coords_text = doc.getElementsByTagName("coordinates")[0].firstChild.data.strip()
    coords_pairs = [c.split(",") for c in coords_text.split()]
    coords = [[float(p[0]), float(p[1])] for p in coords_pairs]
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    return [coords]

kml_path = "polygons/BlackHills.kml"
raw_coords = parse_kml_coordinates(kml_path)
aoi = ee.Geometry.Polygon(raw_coords)
centroid = aoi.centroid().coordinates().getInfo()
center_latlon = [centroid[1], centroid[0]]
print(f"AOI centroid: {center_latlon}")


AOI centroid: [43.988020092438674, -103.73714829146317]


In [ ]:
# 3. Load satellite embeddings
year = 2024
start = ee.Date.fromYMD(year, 1, 1)
end = start.advance(1, "year")

print("Loading Google Satellite Embeddings...")
emb_coll = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
emb_sub = emb_coll.filterDate(start, end).filterBounds(aoi)
count = emb_sub.size().getInfo()
print(f"Number of embedding tiles in AOI ({year}): {count}")

if count == 0:
    print("No embeddings found for this year & AOI. Using first available image instead.")
    emb_img = emb_coll.first().clip(aoi)
else:
    emb_img = emb_sub.mosaic().clip(aoi)

bands = emb_img.bandNames().getInfo()
print("Embedding bands:", bands)

Loading Google Satellite Embeddings...
Number of embedding tiles in AOI (2024): 2
Embedding bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']


In [ ]:
# 4. Visualization
Map = geemap.Map(center=center_latlon, zoom=8)
Map.addLayer(aoi, {"color": "red"}, "AOI")
if len(bands) >= 3:
    vis_params = {"min": -1, "max": 1, "bands": bands[:3]}
    Map.addLayer(emb_img, vis_params, "Embeddings RGB")
Map

Map(center=[43.988020092438674, -103.73714829146317], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
# 5. Sample embeddings for ML
n_samples = 500
print(f"Sampling {n_samples} points from embeddings...")
sample_fc = emb_img.sample(region=aoi, scale=500, numPixels=n_samples, seed=42, geometries=False)
sample = sample_fc.getInfo()
features = np.array([f["properties"] for f in sample["features"]])
X = np.array([[v for v in feat.values()] for feat in features])

print("Sample shape:", X.shape)
print("Embedding stats: mean =", np.mean(X), "std =", np.std(X))

Sampling 500 points from embeddings...
Sample shape: (500, 64)
Embedding stats: mean = 0.01232889657823914 std = 0.12438625239630019
